This is a draft of updating the data collection process to use threads

This code is AI generated, and requires review

In [ ]:

def fetch_single_deck(code):
    """
    This function handles the isolated task of fetching one deck.
    It opens its own browser, tries 3 times, and returns the data.
    """
    # 1. Give this thread its own browser instance
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), 
                              options=chrome_options)
    waiter = WebDriverWait(driver=driver, timeout=30)
    
    DECKLOG_BASE_URL = "https://decklog-en.bushiroad.com/view/" 
    url = DECKLOG_BASE_URL + str(code)
    
    # We use a dictionary to return the results cleanly
    result = {'code': code, 'deckDict': None, 'boss': None, 'success': False}
    
    tries = 0
    while tries < 3:
        tries += 1
        try:
            driver.get(url)
            waiter.until(present((By.CLASS_NAME, "card-controller")))
            html = driver.page_source
            deckSoup = Soup(html, features='html.parser')
            
            # Use your existing parsing function
            deckDict, boss = decklogToDict(deckSoup) 
            
            result['deckDict'] = deckDict
            result['boss'] = boss
            result['success'] = True
            break # Success! Break out of the retry loop
            
        except Exception as e:
            if tries == 3:
                print(f"Failed on deck {code} on try {tries}. Error: {e}")
                
    # Always remember to close the thread's browser!
    driver.quit() 
    return result

def main(base_url, event_info):
    # ... (Part 1 and 1.5 remain exactly the same) ...

    # Part 2: Get deck info from decklog (PARALLELIZED) ~~~~~~~~~~~~~~~~~~~~~
    print("Starting parallel deck fetching...")
    
    # Create a list of only the deck codes we actually need to fetch
    codes_to_fetch = df[df['deck'].isnull()]['decklog'].tolist()
    
    # Setup the Thread Pool
    # max_workers=4 means 4 headless browsers running at the exact same time. 
    # Adjust this up or down depending on your computer's RAM!
    with ThreadPoolExecutor(max_workers=4) as executor:
        
        # Submit all our tasks to the pool
        # This creates a dictionary tracking the 'future' (the pending task) to its code
        futures = {executor.submit(fetch_single_deck, code): code for code in codes_to_fetch}
        
        # as_completed yields results as soon as any thread finishes
        for future in as_completed(futures):
            res = future.result()
            
            if res['success']:
                # Find the correct row in the dataframe and update it
                # Using loc ensures we put the data in exactly the right place
                mask = df['decklog'] == res['code']
                df.loc[mask, 'deck'] = [res['deckDict']] # Wrap dict in list for pandas
                df.loc[mask, 'boss'] = res['boss']
                print(f"Successfully fetched: {res['code']}")

    # ... (Part 2.5 and Part 3 remain exactly the same) ...